# Margin-Template Rule Ensembles: Controlled Ablations and Sensitivity

Runs controlled ablations for Margin-Template Rule Ensembles (MTE).

Evaluates:
1. Reference SMTE baseline.
2. No signed evidence.
3. No log-margin.
4. No mask penalty.
5. Fixed thresholds.
6. No learned template scales.
7. Gaussian versus Student-t (GMTE vs SMTE).
8. Rule-count sensitivity.
9. Mask-penalty sensitivity.

## 0. Dependencies

In [1]:
!pip install -q numpy pandas scikit-learn matplotlib seaborn torch

## 1. Imports and global configuration

In [2]:
import json
import math
import os
import platform
import random
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ----------------------------
# Reproducibility and runtime
# ----------------------------
SEED = 42
SEEDS_FULL = [1, 12, 123, 1234, 12345]

QUICK_MODE = False

if QUICK_MODE:
    SEEDS = [1]
    DATASET_NAMES = ["Hierarchical", "ThresholdCls"]
    EPOCHS = 60
    PATIENCE = 10
else:
    SEEDS = SEEDS_FULL
    DATASET_NAMES = ["Hierarchical", "Friedman1", "XorChain", "ThresholdCls", "MultiThresh"]
    EPOCHS = 150
    PATIENCE = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = Path("outputs_ablation_paper")
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Data and training defaults
# ----------------------------
N_MAIN = 2000
D_MAIN = 15

DEFAULT_N_RULES = 32
BATCH_SIZE = 256
LR = 3e-3
WEIGHT_DECAY = 1e-4
DEFAULT_MASK_L1 = 1e-4
RULE_BETA = 6.0

TRAIN_TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20

# ----------------------------
# Interpretation defaults
# ----------------------------
GRADIENT_N_SAMPLES = 300
TOP_RULES_DISPLAY = 5
MAX_DISPLAY_LITERALS = 4
RULE_DISPLAY_ABSOLUTE_FLOOR = 0.05
RULE_DISPLAY_RELATIVE_FRACTION = 0.20

RULE_COUNTS = [16, 32, 64, 128]
MASK_L1_VALUES = [0.0, 1e-5, 1e-4, 1e-3]

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4f}".format)

MODEL_COLORS = {
    "SMTE-reference": "#2563EB",
    "GMTE-template": "#06B6D4",
    "NoSignedEvidence": "#F97316",
    "NoLogMargin": "#DC2626",
    "NoMaskPenalty": "#16A34A",
    "FixedThresholds": "#7C3AED",
    "FixedScales": "#64748B",
}

SENS_COLORS = {
    "PerfScore": "#2563EB",
    "Composite_I": "#7C3AED",
    "RuleSparsity": "#16A34A",
    "MaskEntropy": "#DC2626",
    "TrainTime": "#F97316",
}


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now() -> float:
    return time.perf_counter()


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mean_std_str(values, digits: int = 3) -> str:
    x = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if len(x) == 0:
        return "nan±nan"
    if len(x) == 1:
        return f"{x.mean():.{digits}f}±nan"
    return f"{x.mean():.{digits}f}±{x.std(ddof=1):.{digits}f}"


def inv_softplus(y: float) -> float:
    y = float(y)
    return math.log(math.exp(y) - 1.0)


def set_paper_style() -> None:
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update(
        {
            "figure.dpi": 300,
            "savefig.dpi": 600,
            "font.family": "serif",
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "legend.fontsize": 8,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


def savefig(name: str) -> None:
    path = FIG_DIR / f"{name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        plt.tight_layout()
    except Exception:
        pass
    plt.savefig(path, bbox_inches="tight")
    plt.close()


set_seed(SEED)
set_paper_style()

print(f"Device      : {DEVICE}")
print(f"Quick mode  : {QUICK_MODE}")
print(f"Seeds       : {SEEDS}")
print(f"Datasets    : {DATASET_NAMES}")
print(f"Output dir  : {OUTPUT_DIR}")

Device      : cuda
Quick mode  : False
Seeds       : [1, 12, 123, 1234, 12345]
Datasets    : ['Hierarchical', 'Friedman1', 'XorChain', 'ThresholdCls', 'MultiThresh']
Output dir  : outputs_ablation_paper


## 2. Synthetic datasets

In [3]:
SYNTHETIC_DATASETS: Dict[str, dict] = {}


def make_hierarchical(n: int = N_MAIN, d: int = D_MAIN, noise: float = 0.3, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.float32)
    y += 3.0 * ((X[:, 0] > 0.5) & (X[:, 1] < 0)).astype(np.float32)
    y += 2.0 * (X[:, 2] > 1).astype(np.float32) * X[:, 5]
    y += 1.5 * X[:, 8]
    y += 0.8 * X[:, 0] * X[:, 1]
    y += noise * rng.standard_normal(n).astype(np.float32)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)


SYNTHETIC_DATASETS["Hierarchical"] = dict(
    loader=make_hierarchical,
    task="regression",
    true_rule="3*1{x0>0.5,x1<0} + 2*1{x2>1}*x5 + 1.5*x8 + 0.8*x0*x1",
    active_feats={0: None, 1: None, 2: None, 5: None, 8: +1.5},
    n_active=5,
)


def make_friedman1(n: int = N_MAIN, d: int = D_MAIN, noise: float = 0.3, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(0, 1, (n, d)).astype(np.float32)

    y = (
        10.0 * np.sin(np.pi * X[:, 0] * X[:, 1])
        + 20.0 * (X[:, 2] - 0.5) ** 2
        + 10.0 * X[:, 3]
        + 5.0 * X[:, 4]
        + noise * rng.standard_normal(n).astype(np.float32)
    ).astype(np.float32)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)


SYNTHETIC_DATASETS["Friedman1"] = dict(
    loader=make_friedman1,
    task="regression",
    true_rule="10*sin(pi*x0*x1) + 20*(x2-0.5)^2 + 10*x3 + 5*x4",
    active_feats={0: None, 1: None, 2: None, 3: +10.0, 4: +5.0},
    n_active=5,
)


def make_xor_chain(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.int64)
    y[((X[:, 0] > 0) ^ (X[:, 1] > 0))] = 1
    y[(X[:, 2] > 0.5) & (X[:, 3] < -0.5)] = 2

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)


SYNTHETIC_DATASETS["XorChain"] = dict(
    loader=make_xor_chain,
    task="classification",
    true_rule="class 1: x0 XOR x1; class 2: x2>0.5 AND x3<-0.5",
    active_feats={0: None, 1: None, 2: None, 3: None},
    n_active=4,
)


def make_threshold_cls(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    score = (
        2.0 * (X[:, 0] > 0.3).astype(np.float32)
        - 1.5 * (X[:, 4] > 0).astype(np.float32)
        + X[:, 7]
    )
    y = (score > score.mean()).astype(np.int64)

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)


SYNTHETIC_DATASETS["ThresholdCls"] = dict(
    loader=make_threshold_cls,
    task="classification",
    true_rule="2*1{x0>0.3} - 1.5*1{x4>0} + x7",
    active_feats={0: None, 4: None, 7: +1.0},
    n_active=3,
)


def make_multithresh(n: int = N_MAIN, d: int = D_MAIN, seed: int = SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d)).astype(np.float32)

    y = np.zeros(n, dtype=np.int64)
    y[(X[:, 0] > 0) & (X[:, 1] > 0)] = 1
    y[(X[:, 0] < 0) & (X[:, 1] > 0) & (X[:, 2] > 0.5)] = 2
    y[(X[:, 6] > 1.0)] = 3

    return pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]), pd.Series(y)


SYNTHETIC_DATASETS["MultiThresh"] = dict(
    loader=make_multithresh,
    task="classification",
    true_rule="class 1: x0>0 AND x1>0; class 2: x0<0 AND x1>0 AND x2>0.5; class 3: x6>1",
    active_feats={0: None, 1: None, 2: None, 6: None},
    n_active=4,
)

dataset_rows = []
for name in DATASET_NAMES:
    meta = SYNTHETIC_DATASETS[name]
    X, y = meta["loader"](seed=SEED)
    dataset_rows.append(
        {
            "Dataset": name,
            "Task": meta["task"],
            "n": X.shape[0],
            "d": X.shape[1],
            "n_active": meta["n_active"],
            "active_features": str([f"x{i}" for i in meta["active_feats"].keys()]),
            "true_rule": meta["true_rule"],
        }
    )

dataset_table = pd.DataFrame(dataset_rows)
dataset_table.to_csv(TABLE_DIR / "table_ablation_dataset_specification.csv", index=False)
display(dataset_table)

,Dataset,Task,n,d,n_active,active_features,true_rule
0,Hierarchical,regression,2000,15,5,"['x0', 'x1', 'x2', 'x5', 'x8']","3*1{x0>0.5,x1<0} + 2*1{x2>1}*x5 + 1.5*x8 + 0.8..."
1,Friedman1,regression,2000,15,5,"['x0', 'x1', 'x2', 'x3', 'x4']",10*sin(pi*x0*x1) + 20*(x2-0.5)^2 + 10*x3 + 5*x4
2,XorChain,classification,2000,15,4,"['x0', 'x1', 'x2', 'x3']",class 1: x0 XOR x1; class 2: x2>0.5 AND x3<-0.5
3,ThresholdCls,classification,2000,15,3,"['x0', 'x4', 'x7']",2*1{x0>0.3} - 1.5*1{x4>0} + x7
4,MultiThresh,classification,2000,15,4,"['x0', 'x1', 'x2', 'x6']",class 1: x0>0 AND x1>0; class 2: x0<0 AND x1>0...


## 3. Data preparation

In [4]:
@dataclass
class PreparedData:
    dataset: str
    seed: int
    task: str
    feature_names: List[str]
    n_classes: int

    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray

    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray

    y_train_scaled: np.ndarray
    y_val_scaled: np.ndarray

    y_scaler: Optional[StandardScaler]
    x_scaler: StandardScaler


def prepare_dataset(ds_name: str, seed: int = SEED) -> PreparedData:
    meta = SYNTHETIC_DATASETS[ds_name]
    X, y = meta["loader"](seed=seed)
    feature_names = list(X.columns)

    strat = y if meta["task"] == "classification" else None

    X_trva, X_te, y_trva, y_te = train_test_split(
        X,
        y,
        test_size=TRAIN_TEST_SIZE,
        random_state=seed,
        stratify=strat,
    )

    strat2 = y_trva if meta["task"] == "classification" else None

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_trva,
        y_trva,
        test_size=VAL_SIZE_WITHIN_TRAIN,
        random_state=seed,
        stratify=strat2,
    )

    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(X_tr).astype(np.float32)
    X_val = x_scaler.transform(X_va).astype(np.float32)
    X_test = x_scaler.transform(X_te).astype(np.float32)

    if meta["task"] == "regression":
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_tr.to_numpy().reshape(-1, 1)).ravel().astype(np.float32)
        y_val_scaled = y_scaler.transform(y_va.to_numpy().reshape(-1, 1)).ravel().astype(np.float32)

        y_train = y_tr.to_numpy().astype(np.float32)
        y_val = y_va.to_numpy().astype(np.float32)
        y_test = y_te.to_numpy().astype(np.float32)

        n_classes = 1
    else:
        y_scaler = None

        y_train = y_tr.to_numpy().astype(np.int64)
        y_val = y_va.to_numpy().astype(np.int64)
        y_test = y_te.to_numpy().astype(np.int64)

        y_train_scaled = y_train.copy()
        y_val_scaled = y_val.copy()

        n_classes = int(len(np.unique(y_train)))

    return PreparedData(
        dataset=ds_name,
        seed=seed,
        task=meta["task"],
        feature_names=feature_names,
        n_classes=n_classes,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        y_train_scaled=y_train_scaled,
        y_val_scaled=y_val_scaled,
        y_scaler=y_scaler,
        x_scaler=x_scaler,
    )

## 4. Ablation configuration

In [5]:
@dataclass
class AblationConfig:
    name: str
    family: str = "SMTE"  # "SMTE" or "GMTE"
    n_rules: int = DEFAULT_N_RULES
    mask_l1: float = DEFAULT_MASK_L1

    use_signed_evidence: bool = True
    use_log_margin: bool = True
    learn_thresholds: bool = True
    learn_scales: bool = True

    beta: float = RULE_BETA
    epochs: int = EPOCHS
    patience: int = PATIENCE
    batch_size: int = BATCH_SIZE
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY

    group: str = "architecture"
    display_label: Optional[str] = None


def architecture_ablation_configs() -> List[AblationConfig]:
    return [
        AblationConfig(
            name="SMTE-reference",
            family="SMTE",
            group="architecture",
            display_label="Reference SMTE",
        ),
        AblationConfig(
            name="NoSignedEvidence",
            family="SMTE",
            use_signed_evidence=False,
            group="architecture",
            display_label="No signed evidence",
        ),
        AblationConfig(
            name="NoLogMargin",
            family="SMTE",
            use_log_margin=False,
            group="architecture",
            display_label="No log-margin",
        ),
        AblationConfig(
            name="NoMaskPenalty",
            family="SMTE",
            mask_l1=0.0,
            group="architecture",
            display_label="No mask penalty",
        ),
        AblationConfig(
            name="FixedThresholds",
            family="SMTE",
            learn_thresholds=False,
            group="architecture",
            display_label="Fixed thresholds",
        ),
        AblationConfig(
            name="FixedScales",
            family="SMTE",
            learn_scales=False,
            group="architecture",
            display_label="Fixed scales",
        ),
        AblationConfig(
            name="GMTE-template",
            family="GMTE",
            group="architecture",
            display_label="Gaussian template",
        ),
    ]


def rule_count_configs() -> List[AblationConfig]:
    out = []
    for r in RULE_COUNTS:
        out.append(
            AblationConfig(
                name=f"SMTE-R{r}",
                family="SMTE",
                n_rules=r,
                group="rule_count",
                display_label=f"R={r}",
            )
        )
    return out


def mask_penalty_configs() -> List[AblationConfig]:
    out = []
    for lam in MASK_L1_VALUES:
        safe = f"{lam:.0e}".replace("+", "").replace("-", "m")
        out.append(
            AblationConfig(
                name=f"SMTE-lambda-{safe}",
                family="SMTE",
                mask_l1=lam,
                group="mask_penalty",
                display_label=fr"$\lambda_m={lam:g}$",
            )
        )
    return out


ALL_CONFIGS = architecture_ablation_configs() + rule_count_configs() + mask_penalty_configs()

config_table = pd.DataFrame([asdict(c) for c in ALL_CONFIGS])
config_table.to_csv(TABLE_DIR / "table_ablation_configurations.csv", index=False)
display(config_table)

,name,family,n_rules,mask_l1,use_signed_evidence,use_log_margin,learn_thresholds,learn_scales,beta,epochs,patience,batch_size,lr,weight_decay,group,display_label
0,SMTE-reference,SMTE,32,0.0001,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,architecture,Reference SMTE
1,NoSignedEvidence,SMTE,32,0.0001,False,True,True,True,6.0000,150,20,256,0.0030,0.0001,architecture,No signed evidence
2,NoLogMargin,SMTE,32,0.0001,True,False,True,True,6.0000,150,20,256,0.0030,0.0001,architecture,No log-margin
3,NoMaskPenalty,SMTE,32,0.0000,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,architecture,No mask penalty
4,FixedThresholds,SMTE,32,0.0001,True,True,False,True,6.0000,150,20,256,0.0030,0.0001,architecture,Fixed thresholds
5,FixedScales,SMTE,32,0.0001,True,True,True,False,6.0000,150,20,256,0.0030,0.0001,architecture,Fixed scales
6,GMTE-template,GMTE,32,0.0001,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,architecture,Gaussian template
7,SMTE-R16,SMTE,16,0.0001,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,rule_count,R=16
8,SMTE-R32,SMTE,32,0.0001,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,rule_count,R=32
9,SMTE-R64,SMTE,64,0.0001,True,True,True,True,6.0000,150,20,256,0.0030,0.0001,rule_count,R=64


## 5. Flexible MTE implementation

In [6]:
@torch.no_grad()
def initialize_thresholds(
    th_tensor: torch.Tensor,
    X: np.ndarray,
    q_lo: float = 0.10,
    q_hi: float = 0.90,
    max_q: int = 9,
) -> None:
    R, D = th_tensor.shape
    X = np.asarray(X, dtype=np.float32)

    for j in range(D):
        qs = np.linspace(q_lo, q_hi, min(R, max_q), dtype=np.float32)
        vals = np.quantile(X[:, j], qs).astype(np.float32)
        vals = np.resize(vals, R)
        th_tensor[:, j].copy_(torch.tensor(vals, dtype=torch.float32, device=th_tensor.device))


def safe_log_margin(u: torch.Tensor, v: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    return torch.log(u + eps) - torch.log(v + eps)


class FlexibleRuleLayer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_rules: int,
        beta: float = RULE_BETA,
        use_signed_evidence: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.n_rules = n_rules
        self.beta = beta
        self.use_signed_evidence = use_signed_evidence

        self.th = nn.Parameter(torch.zeros(n_rules, input_dim))
        self.ineq = nn.Parameter(torch.randn(n_rules, input_dim) * 0.1)

        self.esign = nn.Parameter(torch.randn(n_rules, input_dim) * 0.1)
        self.mask_logit = nn.Parameter(torch.full((n_rules, input_dim), -2.0))

        self.log_kappa = nn.Parameter(torch.tensor(math.log(6.0)))
        self.t = nn.Parameter(torch.zeros(n_rules))

    @torch.no_grad()
    def init_from_data(self, X: np.ndarray) -> None:
        initialize_thresholds(self.th, X)

    def literals(self, x: torch.Tensor):
        kappa = torch.exp(self.log_kappa).clamp(0.5, 50.0)

        direction = torch.tanh(self.ineq)[None, :, :]
        comparison = torch.sigmoid(kappa * direction * (x[:, None, :] - self.th[None, :, :]))

        mask = torch.sigmoid(self.mask_logit)[None, :, :]

        if self.use_signed_evidence:
            sign = torch.tanh(self.esign)[None, :, :]
        else:
            sign = torch.ones_like(mask)

        signed_literal = mask * sign * (2.0 * comparison - 1.0)

        positive_evidence = mask * (
            0.5 * (1.0 + sign) * comparison
            + 0.5 * (1.0 - sign) * (1.0 - comparison)
        )
        negative_evidence = mask - positive_evidence

        return positive_evidence, negative_evidence, signed_literal, mask, comparison, sign


class FlexibleMTE(nn.Module):
    def __init__(self, input_dim: int, n_classes: int, cfg: AblationConfig):
        super().__init__()

        self.cfg = cfg
        self.n_classes = n_classes
        self.output_dim = max(n_classes, 1)

        self.rules = FlexibleRuleLayer(
            input_dim=input_dim,
            n_rules=cfg.n_rules,
            beta=cfg.beta,
            use_signed_evidence=cfg.use_signed_evidence,
        )

        if cfg.family == "SMTE":
            self.mu = nn.Parameter(torch.zeros(cfg.n_rules, input_dim))
            self.log_s = nn.Parameter(torch.zeros(cfg.n_rules, input_dim))
            self.log_nu = nn.Parameter(torch.ones(cfg.n_rules) * math.log(5.0))
        elif cfg.family == "GMTE":
            self.loc = nn.Parameter(torch.zeros(cfg.n_rules, input_dim))
            self.scale = nn.Parameter(torch.full((cfg.n_rules, input_dim), inv_softplus(1.0)))
        else:
            raise ValueError(f"Unknown family: {cfg.family}")

        self.head = nn.Linear(cfg.n_rules, self.output_dim)

    def init_from_data(self, X: np.ndarray) -> None:
        self.rules.init_from_data(X)

        if not self.cfg.learn_thresholds:
            self.rules.th.requires_grad_(False)

        if not self.cfg.learn_scales:
            if self.cfg.family == "SMTE":
                with torch.no_grad():
                    self.log_s.fill_(inv_softplus(1.0))
                self.log_s.requires_grad_(False)
            else:
                with torch.no_grad():
                    self.scale.fill_(inv_softplus(1.0))
                self.scale.requires_grad_(False)

    def nu(self) -> torch.Tensor:
        if self.cfg.family != "SMTE":
            raise RuntimeError("nu() is defined only for SMTE.")
        return F.softplus(self.log_nu).clamp(1.0, 50.0)

    def feature_representation(self, x: torch.Tensor):
        u, v, signed_literal, mask, comparison, sign = self.rules.literals(x)

        if self.cfg.use_log_margin:
            rep = safe_log_margin(u, v)
        else:
            rep = signed_literal

        return rep, signed_literal, mask

    def activations(self, x: torch.Tensor):
        rep, signed_literal, mask = self.feature_representation(x)

        if self.cfg.family == "SMTE":
            nu = self.nu()
            s2 = F.softplus(self.log_s) ** 2 + 1e-6
            energy_per_dim = -((nu[None, :, None] + 1.0) / 2.0) * torch.log1p(
                (rep - self.mu[None, :, :]) ** 2 / (nu[None, :, None] * s2[None, :, :])
            )
            energy = energy_per_dim.sum(dim=2)
        else:
            s2 = F.softplus(self.scale)[None, :, :] ** 2 + 1e-6
            energy = -((rep - self.loc[None, :, :]) ** 2 / (2.0 * s2)).sum(dim=2)

        h = torch.sigmoid(self.rules.beta * (energy - self.rules.t[None, :]))
        return h

    def forward(self, x: torch.Tensor):
        h = self.activations(x)
        return self.head(h)

    def supervised_loss(self, logits: torch.Tensor, y: torch.Tensor):
        if self.n_classes > 1:
            return F.cross_entropy(logits, y.long())
        return F.mse_loss(logits.view(-1), y.float())

    def loss_batch(self, x: torch.Tensor, y: torch.Tensor):
        return self.supervised_loss(self.forward(x), y)


def build_model(pp: PreparedData, cfg: AblationConfig) -> FlexibleMTE:
    model = FlexibleMTE(
        input_dim=pp.X_train.shape[1],
        n_classes=pp.n_classes,
        cfg=cfg,
    ).to(DEVICE)
    model.init_from_data(pp.X_train)
    return model

## 6. Training, prediction, and metrics

In [7]:
def train_model(model: FlexibleMTE, pp: PreparedData, cfg: AblationConfig) -> Dict[str, List[float]]:
    Xtr = torch.tensor(pp.X_train, dtype=torch.float32, device=DEVICE)
    Xva = torch.tensor(pp.X_val, dtype=torch.float32, device=DEVICE)

    if pp.task == "regression":
        ytr = torch.tensor(pp.y_train_scaled, dtype=torch.float32, device=DEVICE)
        yva = torch.tensor(pp.y_val_scaled, dtype=torch.float32, device=DEVICE)
    else:
        ytr = torch.tensor(pp.y_train_scaled, dtype=torch.long, device=DEVICE)
        yva = torch.tensor(pp.y_val_scaled, dtype=torch.long, device=DEVICE)

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)

    best_val = float("inf")
    best_state = None
    wait = 0

    history = {"epoch": [], "train_loss": [], "val_loss": [], "mask_mean": [], "kappa": []}

    for epoch in range(cfg.epochs):
        model.train()

        perm = torch.randperm(len(Xtr), device=DEVICE)
        train_losses = []

        for i in range(0, len(Xtr), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            xb = Xtr[idx]
            yb = ytr[idx]

            opt.zero_grad(set_to_none=True)

            loss = model.loss_batch(xb, yb)
            if cfg.mask_l1 > 0:
                loss = loss + cfg.mask_l1 * torch.sigmoid(model.rules.mask_logit).sum()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            opt.step()
            train_losses.append(float(loss.detach().cpu()))

        sched.step()

        model.eval()
        with torch.no_grad():
            val_loss = float(model.loss_batch(Xva, yva).detach().cpu())
            mask_mean = float(torch.sigmoid(model.rules.mask_logit).mean().detach().cpu())
            kappa = float(torch.exp(model.rules.log_kappa).clamp(0.5, 50.0).detach().cpu())

        history["epoch"].append(epoch)
        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(val_loss)
        history["mask_mean"].append(mask_mean)
        history["kappa"].append(kappa)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return history


@torch.no_grad()
def predict_model(model: FlexibleMTE, pp: PreparedData) -> np.ndarray:
    model.eval()

    X = torch.tensor(pp.X_test, dtype=torch.float32, device=DEVICE)
    out = model(X).detach().cpu().numpy()

    if pp.task == "regression":
        pred_scaled = out.reshape(-1, 1)
        return pp.y_scaler.inverse_transform(pred_scaled).ravel()

    return np.argmax(out, axis=1).astype(int)


def metrics_from_prediction(pp: PreparedData, pred: np.ndarray) -> Dict[str, float]:
    if pp.task == "regression":
        return {
            "RMSE": rmse(pp.y_test, pred),
            "R2": float(r2_score(pp.y_test, pred)),
            "Accuracy": np.nan,
            "F1": np.nan,
        }

    return {
        "RMSE": np.nan,
        "R2": np.nan,
        "Accuracy": float(accuracy_score(pp.y_test, pred)),
        "F1": float(f1_score(pp.y_test, pred, average="macro", zero_division=0)),
    }

## 7. Interpretation diagnostics

In [8]:
def gradient_saliency(model: FlexibleMTE, pp: PreparedData, n_samples: int = GRADIENT_N_SAMPLES):
    model.eval()

    n = min(n_samples, len(pp.X_train))
    X = torch.tensor(pp.X_train[:n], dtype=torch.float32, device=DEVICE).requires_grad_(True)

    with torch.enable_grad():
        out = model(X)
        score = out.sum()
        score.backward()

    grad = X.grad.detach().cpu().numpy()

    sal = np.abs(grad).mean(axis=0)
    signed = grad.mean(axis=0)

    sal = sal / (sal.max() + 1e-12)

    return sal, signed


def gradient_interaction_matrix(model: FlexibleMTE, pp: PreparedData, n_samples: int = GRADIENT_N_SAMPLES):
    model.eval()

    n = min(n_samples, len(pp.X_train))
    X = torch.tensor(pp.X_train[:n], dtype=torch.float32, device=DEVICE).requires_grad_(True)

    with torch.enable_grad():
        out = model(X)
        score = out.sum()
        grads = torch.autograd.grad(score, X)[0]

    G = grads.detach().cpu().numpy()
    Gc = G - G.mean(axis=0, keepdims=True)

    M = np.abs(Gc.T @ Gc) / max(len(Gc), 1)
    np.fill_diagonal(M, 0.0)

    M = M / (M.max() + 1e-12)
    return M


def rule_mask_matrix(model: FlexibleMTE) -> np.ndarray:
    return torch.sigmoid(model.rules.mask_logit).detach().cpu().numpy()


def top_rule_indices(model: FlexibleMTE, k: int = TOP_RULES_DISPLAY):
    w = model.head.weight.detach().cpu().numpy()
    imp = np.abs(w).sum(axis=0)
    idx = np.argsort(imp)[::-1][:k]
    return idx, imp


def extract_rules(model: FlexibleMTE, pp: PreparedData, top_k: int = TOP_RULES_DISPLAY) -> List[Dict[str, Any]]:
    rules = model.rules

    head_w = model.head.weight.detach().cpu().numpy()
    rule_importance = np.abs(head_w).sum(axis=0)

    mask = torch.sigmoid(rules.mask_logit).detach().cpu().numpy()
    th_scaled = rules.th.detach().cpu().numpy()
    direction = torch.tanh(rules.ineq).detach().cpu().numpy()

    if rules.use_signed_evidence:
        evidence_sign = torch.tanh(rules.esign).detach().cpu().numpy()
    else:
        evidence_sign = np.ones_like(direction)

    scaler_mean = pp.x_scaler.mean_
    scaler_std = pp.x_scaler.scale_
    th_raw = th_scaled * scaler_std[None, :] + scaler_mean[None, :]

    selected = np.argsort(rule_importance)[::-1][:top_k]
    out = []

    for rank, r in enumerate(selected, start=1):
        cutoff = max(RULE_DISPLAY_ABSOLUTE_FLOOR, RULE_DISPLAY_RELATIVE_FRACTION * float(mask[r].max()))

        active = np.where(mask[r] > cutoff)[0]
        active = active[np.argsort(mask[r, active])[::-1]]
        active = active[:MAX_DISPLAY_LITERALS]

        conditions = []
        for j in active:
            op = ">" if direction[r, j] > 0 else "<"
            ev = "+" if evidence_sign[r, j] > 0 else "-"

            conditions.append(
                {
                    "feature": pp.feature_names[j],
                    "feature_index": int(j),
                    "operator": op,
                    "threshold_raw": float(th_raw[r, j]),
                    "threshold_scaled": float(th_scaled[r, j]),
                    "evidence": ev,
                    "mask": float(mask[r, j]),
                    "text": (
                        f"{pp.feature_names[j]} {op} {th_raw[r, j]:+.3f} "
                        f"(evidence={ev}, mask={mask[r, j]:.2f})"
                    ),
                }
            )

        out.append(
            {
                "rank": rank,
                "rule_id": int(r),
                "importance": float(rule_importance[r]),
                "max_mask": float(mask[r].max()),
                "cutoff": float(cutoff),
                "n_conditions": int(len(conditions)),
                "conditions": conditions,
                "rule_text": " AND ".join([c["text"] for c in conditions])
                if conditions
                else "(no literal above display threshold)",
            }
        )

    return out


def mask_summary(model: FlexibleMTE, top_k: int = TOP_RULES_DISPLAY) -> Dict[str, float]:
    mask = rule_mask_matrix(model)
    entropy = -mask * np.log(mask + 1e-12) - (1.0 - mask) * np.log(1.0 - mask + 1e-12)

    idx, _ = top_rule_indices(model, k=top_k)

    displayed_counts = []
    for r in idx:
        cutoff = max(RULE_DISPLAY_ABSOLUTE_FLOOR, RULE_DISPLAY_RELATIVE_FRACTION * float(mask[r].max()))
        active = np.where(mask[r] > cutoff)[0]
        active = active[np.argsort(mask[r, active])[::-1]]
        active = active[:MAX_DISPLAY_LITERALS]
        displayed_counts.append(len(active))

    out = {
        "MaskMean": float(mask.mean()),
        "MaskMaxMean": float(mask.max(axis=1).mean()),
        "MaskEntropy": float(entropy.mean()),
        "DisplayedTopRuleConditions": int(np.sum(displayed_counts)),
        "MeanDisplayedConditions": float(np.mean(displayed_counts)) if displayed_counts else np.nan,
        "RuleSparsity": float(1.0 - np.mean(displayed_counts) / mask.shape[1]) if displayed_counts else np.nan,
    }

    for th in [0.05, 0.10, 0.25, 0.50]:
        active = mask > th
        suffix = str(th).replace(".", "p")
        out[f"StrongLiteralsGT{suffix}"] = int(active.sum())
        out[f"MeanStrongLiteralsPerRuleGT{suffix}"] = float(active.sum(axis=1).mean())
        out[f"RulesWithAnyLiteralGT{suffix}"] = int((active.sum(axis=1) > 0).sum())

    return out


def interpret_model(model: FlexibleMTE, pp: PreparedData) -> Dict[str, Any]:
    fi, signed = gradient_saliency(model, pp)
    inter = gradient_interaction_matrix(model, pp)
    rules = extract_rules(model, pp)
    ms = mask_summary(model)

    extra = {}
    if model.cfg.family == "SMTE":
        nu = F.softplus(model.log_nu).detach().cpu().numpy()
        extra.update(
            {
                "NuMean": float(nu.mean()),
                "NuMin": float(nu.min()),
                "NuMax": float(nu.max()),
                "TemplateScaleMean": float(F.softplus(model.log_s).detach().cpu().numpy().mean()),
            }
        )
    else:
        extra.update(
            {
                "NuMean": np.nan,
                "NuMin": np.nan,
                "NuMax": np.nan,
                "TemplateScaleMean": float(F.softplus(model.scale).detach().cpu().numpy().mean()),
            }
        )

    return {
        "feature_importance": fi,
        "signed_importance": signed,
        "interaction_matrix": inter,
        "rules": rules,
        **ms,
        **extra,
    }


def compute_interpretability_scores(ds_name: str, pp: PreparedData, interp: Dict[str, Any]) -> Dict[str, float]:
    meta = SYNTHETIC_DATASETS[ds_name]
    active = set(meta["active_feats"].keys())
    D = len(pp.feature_names)

    fi = np.asarray(interp.get("feature_importance", np.zeros(D)), dtype=float)
    top = set(np.argsort(fi)[::-1][:meta["n_active"]])

    precision = len(top & active) / max(len(top), 1)
    recall = len(top & active) / max(len(active), 1)
    fsel_f1 = 2 * precision * recall / (precision + recall + 1e-12)

    rules = interp.get("rules", [])
    if rules and all("n_conditions" in r for r in rules):
        avg_conditions = float(np.mean([r["n_conditions"] for r in rules]))
        rule_sparsity = max(0.0, 1.0 - avg_conditions / D)
    else:
        avg_conditions = np.nan
        rule_sparsity = 0.5

    M = interp.get("interaction_matrix", None)
    af = sorted(active)
    gt_pairs = set((a, b) for i, a in enumerate(af) for b in af[i + 1:])

    if M is not None and len(gt_pairs) > 0 and np.nanmax(M) > 0:
        pairs = sorted(
            [(M[i, j], i, j) for i in range(D) for j in range(i + 1, D)],
            reverse=True,
        )
        pred_pairs = set((min(i, j), max(i, j)) for _, i, j in pairs[:len(gt_pairs)])
        interaction_score = len(pred_pairs & gt_pairs) / len(gt_pairs)
    else:
        pred_pairs = set()
        interaction_score = 0.5

    signed = interp.get("signed_importance", None)
    correct, total = 0, 0
    if signed is not None:
        for j, known_sign in meta["active_feats"].items():
            if known_sign is not None:
                total += 1
                correct += int(np.sign(signed[j]) == np.sign(known_sign))

    direction_score = correct / total if total > 0 else 0.5

    composite = (
        0.40 * fsel_f1
        + 0.25 * rule_sparsity
        + 0.20 * interaction_score
        + 0.15 * direction_score
    )

    return {
        "FeatSel_F1": float(fsel_f1),
        "FeatSel_Precision": float(precision),
        "FeatSel_Recall": float(recall),
        "RuleSparsity": float(rule_sparsity),
        "MeanDisplayedConditions": float(avg_conditions) if not np.isnan(avg_conditions) else np.nan,
        "InteractDetect": float(interaction_score),
        "Direction": float(direction_score),
        "Composite_I": float(composite),
        "SelectedFeatures": str(sorted(list(top))),
        "PredictedInteractionPairs": str(sorted(list(pred_pairs))),
    }

## 8. Controlled ablation run

In [9]:
results = []
failed_rows = []
rule_rows = []
history_rows = []

prepared_cache: Dict[Tuple[int, str], PreparedData] = {}

total_runs = len(SEEDS) * len(DATASET_NAMES) * len(ALL_CONFIGS)
run_id = 0

for seed in SEEDS:
    set_seed(seed)

    for ds_name in DATASET_NAMES:
        if (seed, ds_name) not in prepared_cache:
            prepared_cache[(seed, ds_name)] = prepare_dataset(ds_name, seed=seed)

        pp = prepared_cache[(seed, ds_name)]

        print("\n" + "=" * 110)
        print(f"Seed={seed} | Dataset={ds_name} | Task={pp.task}")
        print(f"Ground truth: {SYNTHETIC_DATASETS[ds_name]['true_rule']}")
        print("=" * 110)

        for cfg in ALL_CONFIGS:
            run_id += 1
            t0 = now()

            try:
                model = build_model(pp, cfg)
                hist = train_model(model, pp, cfg)
                pred = predict_model(model, pp)

                metrics = metrics_from_prediction(pp, pred)
                interp = interpret_model(model, pp)
                scores = compute_interpretability_scores(ds_name, pp, interp)

                elapsed = now() - t0
                params = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
                params_total = int(sum(p.numel() for p in model.parameters()))

                row = {
                    "RunID": run_id,
                    "Seed": seed,
                    "Dataset": ds_name,
                    "Task": pp.task,
                    "Config": cfg.name,
                    "Label": cfg.display_label or cfg.name,
                    "Group": cfg.group,
                    "Family": cfg.family,
                    "Rules": cfg.n_rules,
                    "MaskL1": cfg.mask_l1,
                    "UseSignedEvidence": cfg.use_signed_evidence,
                    "UseLogMargin": cfg.use_log_margin,
                    "LearnThresholds": cfg.learn_thresholds,
                    "LearnScales": cfg.learn_scales,
                    "TrainTime": elapsed,
                    "ParamsTrainable": params,
                    "ParamsTotal": params_total,
                    "EpochsRun": len(hist["epoch"]),
                    "BestValLoss": float(np.min(hist["val_loss"])) if len(hist["val_loss"]) else np.nan,
                    **metrics,
                    **scores,
                    "MaskMean": interp.get("MaskMean", np.nan),
                    "MaskMaxMean": interp.get("MaskMaxMean", np.nan),
                    "MaskEntropy": interp.get("MaskEntropy", np.nan),
                    "DisplayedTopRuleConditions": interp.get("DisplayedTopRuleConditions", np.nan),
                    "MeanDisplayedConditions_FromMask": interp.get("MeanDisplayedConditions", np.nan),
                    "RuleSparsity_FromMask": interp.get("RuleSparsity", np.nan),
                    "NuMean": interp.get("NuMean", np.nan),
                    "NuMin": interp.get("NuMin", np.nan),
                    "NuMax": interp.get("NuMax", np.nan),
                    "TemplateScaleMean": interp.get("TemplateScaleMean", np.nan),
                }

                for key, value in interp.items():
                    if isinstance(value, (int, float, np.integer, np.floating)) and key not in row:
                        row[key] = value

                results.append(row)

                for e, tr, va, mm, kk in zip(
                    hist["epoch"],
                    hist["train_loss"],
                    hist["val_loss"],
                    hist["mask_mean"],
                    hist["kappa"],
                ):
                    history_rows.append(
                        {
                            "Seed": seed,
                            "Dataset": ds_name,
                            "Config": cfg.name,
                            "Group": cfg.group,
                            "Epoch": e,
                            "TrainLoss": tr,
                            "ValLoss": va,
                            "MaskMean": mm,
                            "Kappa": kk,
                        }
                    )

                for r in interp["rules"]:
                    rule_rows.append(
                        {
                            "Seed": seed,
                            "Dataset": ds_name,
                            "Task": pp.task,
                            "Config": cfg.name,
                            "Group": cfg.group,
                            "Family": cfg.family,
                            "Rank": r["rank"],
                            "RuleID": r["rule_id"],
                            "Importance": r["importance"],
                            "MaxMask": r["max_mask"],
                            "Cutoff": r["cutoff"],
                            "NConditions": r["n_conditions"],
                            "Rule": r["rule_text"],
                        }
                    )

                metric_text = (
                    f"RMSE={metrics['RMSE']:.3f} R2={metrics['R2']:.3f}"
                    if pp.task == "regression"
                    else f"Acc={metrics['Accuracy']:.3f} F1={metrics['F1']:.3f}"
                )
                print(
                    f"[{run_id:04d}/{total_runs:04d}] {cfg.name:20s} | "
                    f"{metric_text} | I={scores['Composite_I']:.3f} | "
                    f"FeatF1={scores['FeatSel_F1']:.3f} | "
                    f"Sparse={scores['RuleSparsity']:.3f} | "
                    f"t={elapsed:.1f}s"
                )

            except Exception as e:
                elapsed = now() - t0
                err = f"{type(e).__name__}: {str(e)[:300]}"
                print(f"[{run_id:04d}/{total_runs:04d}] {cfg.name:20s} | FAILED: {err}")

                failed_rows.append(
                    {
                        "RunID": run_id,
                        "Seed": seed,
                        "Dataset": ds_name,
                        "Task": pp.task,
                        "Config": cfg.name,
                        "Group": cfg.group,
                        "Family": cfg.family,
                        "TrainTime": elapsed,
                        "Error": err,
                    }
                )

df_raw = pd.DataFrame(results)
df_failed = pd.DataFrame(failed_rows)
df_rules = pd.DataFrame(rule_rows)
df_history = pd.DataFrame(history_rows)

df_raw.to_csv(TABLE_DIR / "table_ablation_raw_results.csv", index=False)
df_failed.to_csv(TABLE_DIR / "table_ablation_failed_runs.csv", index=False)
df_rules.to_csv(TABLE_DIR / "table_ablation_extracted_rules.csv", index=False)
df_history.to_csv(TABLE_DIR / "table_ablation_training_curves.csv", index=False)

display(df_raw.head())
display(df_failed)


Seed=1 | Dataset=Hierarchical | Task=regression
Ground truth: 3*1{x0>0.5,x1<0} + 2*1{x2>1}*x5 + 1.5*x8 + 0.8*x0*x1
[0001/0375] SMTE-reference       | RMSE=0.642 R2=0.933 | I=0.923 | FeatF1=1.000 | Sparse=0.773 | t=9.3s
[0002/0375] NoSignedEvidence     | RMSE=0.669 R2=0.927 | I=0.873 | FeatF1=1.000 | Sparse=0.733 | t=2.8s
[0003/0375] NoLogMargin          | RMSE=1.052 R2=0.820 | I=0.913 | FeatF1=1.000 | Sparse=0.733 | t=2.9s
[0004/0375] NoMaskPenalty        | RMSE=0.650 R2=0.931 | I=0.933 | FeatF1=1.000 | Sparse=0.733 | t=3.0s
[0005/0375] FixedThresholds      | RMSE=0.695 R2=0.921 | I=0.887 | FeatF1=1.000 | Sparse=0.787 | t=3.1s
[0006/0375] FixedScales          | RMSE=0.705 R2=0.919 | I=0.930 | FeatF1=1.000 | Sparse=0.800 | t=3.0s
[0007/0375] GMTE-template        | RMSE=0.665 R2=0.928 | I=0.917 | FeatF1=1.000 | Sparse=0.827 | t=2.7s
[0008/0375] SMTE-R16             | RMSE=1.003 R2=0.836 | I=0.917 | FeatF1=1.000 | Sparse=0.747 | t=3.1s
[0009/0375] SMTE-R32             | RMSE=0.728 R2=0.9

,RunID,Seed,Dataset,Task,Config,Label,Group,Family,Rules,MaskL1,UseSignedEvidence,UseLogMargin,LearnThresholds,LearnScales,TrainTime,ParamsTrainable,ParamsTotal,EpochsRun,BestValLoss,RMSE,R2,Accuracy,F1,FeatSel_F1,FeatSel_Precision,FeatSel_Recall,RuleSparsity,MeanDisplayedConditions,InteractDetect,Direction,Composite_I,SelectedFeatures,PredictedInteractionPairs,MaskMean,MaskMaxMean,MaskEntropy,DisplayedTopRuleConditions,MeanDisplayedConditions_FromMask,RuleSparsity_FromMask,NuMean,NuMin,NuMax,TemplateScaleMean,StrongLiteralsGT0p05,MeanStrongLiteralsPerRuleGT0p05,RulesWithAnyLiteralGT0p05,StrongLiteralsGT0p1,MeanStrongLiteralsPerRuleGT0p1,RulesWithAnyLiteralGT0p1,StrongLiteralsGT0p25,MeanStrongLiteralsPerRuleGT0p25,RulesWithAnyLiteralGT0p25,StrongLiteralsGT0p5,MeanStrongLiteralsPerRuleGT0p5,RulesWithAnyLiteralGT0p5
0,1,1,Hierarchical,regression,SMTE-reference,Reference SMTE,architecture,SMTE,32,0.0001,True,True,True,True,9.3297,2978,2978,150,0.0710,0.6420,0.9329,NaN,NaN,1.0000,1.0000,1.0000,0.7733,3.4000,0.9000,1.0000,0.9233,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 5), (0, 8), (1, 2), (1, 5...",0.0504,0.0567,0.1996,17,3.4000,0.7733,1.6849,1.5036,1.8474,0.6903,111,3.4688,32,0,0.0000,0,0,0.0000,0,0,0.0000,0
1,2,1,Hierarchical,regression,NoSignedEvidence,No signed evidence,architecture,SMTE,32,0.0001,False,True,True,True,2.8495,2978,2978,142,0.0771,0.6691,0.9271,NaN,NaN,1.0000,1.0000,1.0000,0.7333,4.0000,0.7000,1.0000,0.8733,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 8), (1, 2), (1, 5), (1, 8), (1, 1...",0.0510,0.0567,0.2013,20,4.0000,0.7333,1.9264,1.7815,2.1472,0.7510,210,6.5625,28,0,0.0000,0,0,0.0000,0,0,0.0000,0
2,3,1,Hierarchical,regression,NoLogMargin,No log-margin,architecture,SMTE,32,0.0001,True,False,True,True,2.8631,2978,2978,150,0.1745,1.0523,0.8198,NaN,NaN,1.0000,1.0000,1.0000,0.7333,4.0000,0.9000,1.0000,0.9133,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 5), (0, 8), (1, 5), (1, 8...",0.0910,0.2206,0.2877,20,4.0000,0.7333,1.4893,1.3592,1.6998,0.6494,338,10.5625,32,137,4.2812,32,9,0.2812,8,0,0.0000,0
3,4,1,Hierarchical,regression,NoMaskPenalty,No mask penalty,architecture,SMTE,32,0.0000,True,True,True,True,2.9682,2978,2978,150,0.0763,0.6500,0.9312,NaN,NaN,1.0000,1.0000,1.0000,0.7333,4.0000,1.0000,1.0000,0.9333,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 5), (0, 8), (1, 2), (1, 5...",0.1208,0.1484,0.3680,20,4.0000,0.7333,1.7163,1.5352,1.9671,0.6892,480,15.0000,32,480,15.0000,32,0,0.0000,0,0,0.0000,0
4,5,1,Hierarchical,regression,FixedThresholds,Fixed thresholds,architecture,SMTE,32,0.0001,True,True,False,True,3.0552,2498,2978,150,0.0838,0.6953,0.9213,NaN,NaN,1.0000,1.0000,1.0000,0.7867,3.2000,0.7000,1.0000,0.8867,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 5), (0, 8), (0, 11), (1, ...",0.0504,0.0588,0.1997,16,3.2000,0.7867,1.6686,1.3253,1.8438,0.6865,97,3.0312,32,0,0.0000,0,0,0.0000,0,0,0.0000,0


""


## 9. Normalize performance and compute ranks

In [10]:
def normalized_performance_score(sub: pd.DataFrame, task: str) -> pd.DataFrame:
    sub = sub.copy()

    if task == "regression":
        x = sub["RMSE"].astype(float)
        sub["PerfScore"] = 1.0 - (x - x.min()) / (x.max() - x.min() + 1e-12)
        sub["Rank"] = x.rank(ascending=True, method="average")
    else:
        x = sub["F1"].astype(float)
        sub["PerfScore"] = (x - x.min()) / (x.max() - x.min() + 1e-12)
        sub["Rank"] = x.rank(ascending=False, method="average")

    return sub


scored_blocks = []

for (seed, ds_name, group), sub in df_raw.groupby(["Seed", "Dataset", "Group"]):
    task = sub["Task"].iloc[0]
    scored_blocks.append(normalized_performance_score(sub, task))

df_score = pd.concat(scored_blocks, ignore_index=True) if scored_blocks else pd.DataFrame()
df_score.to_csv(TABLE_DIR / "table_ablation_ranked_results.csv", index=False)

display(df_score.head())

,RunID,Seed,Dataset,Task,Config,Label,Group,Family,Rules,MaskL1,UseSignedEvidence,UseLogMargin,LearnThresholds,LearnScales,TrainTime,ParamsTrainable,ParamsTotal,EpochsRun,BestValLoss,RMSE,R2,Accuracy,F1,FeatSel_F1,FeatSel_Precision,FeatSel_Recall,RuleSparsity,MeanDisplayedConditions,InteractDetect,Direction,Composite_I,SelectedFeatures,PredictedInteractionPairs,MaskMean,MaskMaxMean,MaskEntropy,DisplayedTopRuleConditions,MeanDisplayedConditions_FromMask,RuleSparsity_FromMask,NuMean,NuMin,NuMax,TemplateScaleMean,StrongLiteralsGT0p05,MeanStrongLiteralsPerRuleGT0p05,RulesWithAnyLiteralGT0p05,StrongLiteralsGT0p1,MeanStrongLiteralsPerRuleGT0p1,RulesWithAnyLiteralGT0p1,StrongLiteralsGT0p25,MeanStrongLiteralsPerRuleGT0p25,RulesWithAnyLiteralGT0p25,StrongLiteralsGT0p5,MeanStrongLiteralsPerRuleGT0p5,RulesWithAnyLiteralGT0p5,PerfScore,Rank
0,16,1,Friedman1,regression,SMTE-reference,Reference SMTE,architecture,SMTE,32,0.0001,True,True,True,True,3.0446,2978,2978,150,0.0135,0.5407,0.9861,NaN,NaN,1.0000,1.0000,1.0000,0.8000,3.0000,0.8000,1.0000,0.9100,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4...",0.0501,0.0568,0.1988,15,3.0000,0.8000,1.7413,1.6427,1.8561,0.6908,81,2.5312,31,0,0.0000,0,0,0.0000,0,0,0.0000,0,0.7728,4.0000
1,17,1,Friedman1,regression,NoSignedEvidence,No signed evidence,architecture,SMTE,32,0.0001,False,True,True,True,2.8411,2978,2978,146,0.0067,0.3865,0.9929,NaN,NaN,1.0000,1.0000,1.0000,0.7867,3.2000,0.7000,1.0000,0.8867,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 4), (0, 10), (0, 13), (1,...",0.0508,0.0562,0.2008,16,3.2000,0.7867,1.9289,1.7627,2.0622,0.7476,110,3.4375,19,0,0.0000,0,0,0.0000,0,0,0.0000,0,1.0000,1.0000
2,18,1,Friedman1,regression,NoLogMargin,No log-margin,architecture,SMTE,32,0.0001,True,False,True,True,2.8787,2978,2978,150,0.0485,1.0653,0.9459,NaN,NaN,1.0000,1.0000,1.0000,0.7333,4.0000,0.9000,1.0000,0.9133,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3...",0.0918,0.2134,0.2869,20,4.0000,0.7333,1.4683,1.3046,1.5578,0.6422,298,9.3125,32,149,4.6562,32,5,0.1562,3,0,0.0000,0,0.0000,7.0000
3,19,1,Friedman1,regression,NoMaskPenalty,No mask penalty,architecture,SMTE,32,0.0000,True,True,True,True,3.0088,2978,2978,150,0.0145,0.5495,0.9856,NaN,NaN,1.0000,1.0000,1.0000,0.7333,4.0000,1.0000,1.0000,0.9333,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3...",0.1203,0.1471,0.3670,20,4.0000,0.7333,1.7432,1.6222,1.8731,0.6936,480,15.0000,32,480,15.0000,32,0,0.0000,0,0,0.0000,0,0.7598,5.0000
4,20,1,Friedman1,regression,FixedThresholds,Fixed thresholds,architecture,SMTE,32,0.0001,True,True,False,True,3.0190,2498,2978,150,0.0114,0.4850,0.9888,NaN,NaN,1.0000,1.0000,1.0000,0.7867,3.2000,0.9000,1.0000,0.9267,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3...",0.0502,0.0561,0.1992,16,3.2000,0.7867,1.7251,1.6097,1.8154,0.6914,93,2.9062,32,0,0.0000,0,0,0.0000,0,0,0.0000,0,0.8549,2.0000


## 10. Summary tables

In [11]:
SUMMARY_METRICS = [
    "PerfScore",
    "Rank",
    "TrainTime",
    "EpochsRun",
    "BestValLoss",
    "Composite_I",
    "FeatSel_F1",
    "RuleSparsity",
    "MeanDisplayedConditions",
    "InteractDetect",
    "Direction",
    "RMSE",
    "R2",
    "Accuracy",
    "F1",
    "MaskMean",
    "MaskEntropy",
    "DisplayedTopRuleConditions",
    "NuMean",
    "TemplateScaleMean",
    "ParamsTrainable",
]


def summarize_by(df: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    rows = []
    for key_vals, g in df.groupby(keys):
        if not isinstance(key_vals, tuple):
            key_vals = (key_vals,)

        row = {k: v for k, v in zip(keys, key_vals)}
        row["Runs"] = len(g)

        for c in SUMMARY_METRICS:
            if c in g.columns:
                vals = pd.to_numeric(g[c], errors="coerce")
                row[f"{c}_mean"] = vals.mean()
                row[f"{c}_std"] = vals.std(ddof=1)
                row[f"{c}_mean_std"] = mean_std_str(vals)
            else:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_std"] = np.nan
                row[f"{c}_mean_std"] = "nan±nan"

        rows.append(row)

    return pd.DataFrame(rows)


summary_config = summarize_by(df_score, ["Group", "Config", "Label", "Family", "Rules", "MaskL1"])
summary_config["PaperScore"] = 0.60 * summary_config["PerfScore_mean"] + 0.40 * summary_config["Composite_I_mean"]
summary_config = summary_config.sort_values(["Group", "PaperScore"], ascending=[True, False])
summary_config.to_csv(TABLE_DIR / "table_ablation_summary_by_config.csv", index=False)

summary_dataset = summarize_by(df_score, ["Group", "Dataset", "Task", "Config", "Label"])
summary_dataset.to_csv(TABLE_DIR / "table_ablation_summary_by_dataset.csv", index=False)

arch_summary = summary_config[summary_config["Group"] == "architecture"].copy()
rule_summary = summary_config[summary_config["Group"] == "rule_count"].copy()
mask_summary_df = summary_config[summary_config["Group"] == "mask_penalty"].copy()

arch_summary.to_csv(TABLE_DIR / "paper_table_architecture_ablations.csv", index=False)
rule_summary.to_csv(TABLE_DIR / "paper_table_rule_count_sensitivity.csv", index=False)
mask_summary_df.to_csv(TABLE_DIR / "paper_table_mask_penalty_sensitivity.csv", index=False)

display(arch_summary)
display(rule_summary)
display(mask_summary_df)

,Group,Config,Label,Family,Rules,MaskL1,Runs,PerfScore_mean,PerfScore_std,PerfScore_mean_std,Rank_mean,Rank_std,Rank_mean_std,TrainTime_mean,TrainTime_std,TrainTime_mean_std,EpochsRun_mean,EpochsRun_std,EpochsRun_mean_std,BestValLoss_mean,BestValLoss_std,BestValLoss_mean_std,Composite_I_mean,Composite_I_std,Composite_I_mean_std,FeatSel_F1_mean,FeatSel_F1_std,FeatSel_F1_mean_std,RuleSparsity_mean,RuleSparsity_std,RuleSparsity_mean_std,MeanDisplayedConditions_mean,MeanDisplayedConditions_std,MeanDisplayedConditions_mean_std,InteractDetect_mean,InteractDetect_std,InteractDetect_mean_std,Direction_mean,Direction_std,Direction_mean_std,RMSE_mean,RMSE_std,RMSE_mean_std,R2_mean,R2_std,R2_mean_std,Accuracy_mean,Accuracy_std,Accuracy_mean_std,F1_mean,F1_std,F1_mean_std,MaskMean_mean,MaskMean_std,MaskMean_mean_std,MaskEntropy_mean,MaskEntropy_std,MaskEntropy_mean_std,DisplayedTopRuleConditions_mean,DisplayedTopRuleConditions_std,DisplayedTopRuleConditions_mean_std,NuMean_mean,NuMean_std,NuMean_mean_std,TemplateScaleMean_mean,TemplateScaleMean_std,TemplateScaleMean_mean_std,ParamsTrainable_mean,ParamsTrainable_std,ParamsTrainable_mean_std,PaperScore
4,architecture,NoMaskPenalty,No mask penalty,SMTE,32,0.0000,25,0.8767,0.1516,0.877±0.152,2.5200,1.2117,2.520±1.212,2.9779,0.0352,2.978±0.035,150.0000,0.0000,150.000±0.000,0.0632,0.0356,0.063±0.036,0.8753,0.0473,0.875±0.047,1.0000,0.0000,1.000±0.000,0.7333,0.0000,0.733±0.000,4.0000,0.0000,4.000±0.000,0.8600,0.1330,0.860±0.133,0.8000,0.2500,0.800±0.250,0.5789,0.0715,0.579±0.071,0.9575,0.0320,0.958±0.032,0.9840,0.0116,0.984±0.012,0.9821,0.0117,0.982±0.012,0.1204,0.0003,0.120±0.000,0.3673,0.0007,0.367±0.001,20.0000,0.0000,20.000±0.000,1.7911,0.0775,1.791±0.078,0.6921,0.0048,0.692±0.005,3017.6000,39.2779,3017.600±39.278,0.8762
2,architecture,GMTE-template,Gaussian template,GMTE,32,0.0001,25,0.8274,0.1906,0.827±0.191,3.2600,1.4728,3.260±1.473,2.7790,0.0400,2.779±0.040,150.0000,0.0000,150.000±0.000,0.0766,0.0548,0.077±0.055,0.8812,0.0766,0.881±0.077,0.9800,0.1000,0.980±0.100,0.8427,0.0283,0.843±0.028,2.3600,0.4243,2.360±0.424,0.8227,0.1377,0.823±0.138,0.7600,0.2930,0.760±0.293,0.6466,0.1601,0.647±0.160,0.9511,0.0321,0.951±0.032,0.9822,0.0136,0.982±0.014,0.9812,0.0131,0.981±0.013,0.0502,0.0003,0.050±0.000,0.1989,0.0008,0.199±0.001,11.8000,2.1213,11.800±2.121,NaN,NaN,nan±nan,0.9823,0.0041,0.982±0.004,2985.6000,39.2779,2985.600±39.278,0.8489
6,architecture,SMTE-reference,Reference SMTE,SMTE,32,0.0001,25,0.8245,0.1842,0.824±0.184,3.2800,1.6335,3.280±1.634,3.3448,1.2477,3.345±1.248,150.0000,0.0000,150.000±0.000,0.0654,0.0386,0.065±0.039,0.8809,0.0675,0.881±0.068,1.0000,0.0000,1.000±0.000,0.8187,0.0422,0.819±0.042,2.7200,0.6325,2.720±0.632,0.8413,0.1901,0.841±0.190,0.7200,0.3253,0.720±0.325,0.5760,0.0784,0.576±0.078,0.9580,0.0314,0.958±0.031,0.9800,0.0131,0.980±0.013,0.9791,0.0115,0.979±0.012,0.0502,0.0002,0.050±0.000,0.1991,0.0005,0.199±0.001,13.6000,3.1623,13.600±3.162,1.7892,0.0807,1.789±0.081,0.6919,0.0040,0.692±0.004,3017.6000,39.2779,3017.600±39.278,0.8471
0,architecture,FixedScales,Fixed scales,SMTE,32,0.0001,25,0.7330,0.1873,0.733±0.187,4.5000,1.5000,4.500±1.500,3.0077,0.0476,3.008±0.048,150.0000,0.0000,150.000±0.000,0.0861,0.0665,0.086±0.067,0.8748,0.0667,0.875±0.067,0.9900,0.0500,0.990±0.050,0.8272,0.0307,0.827±0.031,2.5920,0.4600,2.592±0.460,0.8200,0.1528,0.820±0.153,0.7200,0.3253,0.720±0.325,0.6929,0.0801,0.693±0.080,0.9457,0.0347,0.946±0.035,0.9798,0.0135,0.980±0.013,0.9789,0.0123,0.979±0.012,0.0503,0.0002,0.050±0.000,0.1993,0.0007,0.199±0.001,12.9600,2.3000,12.960±2.300,1.6309,0.0387,1.631±0.039,1.0000,0.0000,1.000±0.000,2537.6000,39.2779,2537.600±39.278,0.7897
1,architecture,FixedThresholds,Fixed thresholds,SMTE,32,0.0001,25,0.7259,0.2055,0.726±0.206,4.6200,1.2852,4.620±1.285,3.0315,0.0710,3.031±0.071,149.7600,1.0116,149.760±1.012,0.0859,0.0522,0.086±0.052,0.8592,0.0540,0.859±0.054,1.0000,0.0000,1.000±0.000,0.8101,0.0382,0.810±0.038,2.8480,0.5724,2.848±0.572,0.7733,0.1774,0.773±0.177,0.6800

,Group,Config,Label,Family,Rules,MaskL1,Runs,PerfScore_mean,PerfScore_std,PerfScore_mean_std,Rank_mean,Rank_std,Rank_mean_std,TrainTime_mean,TrainTime_std,TrainTime_mean_std,EpochsRun_mean,EpochsRun_std,EpochsRun_mean_std,BestValLoss_mean,BestValLoss_std,BestValLoss_mean_std,Composite_I_mean,Composite_I_std,Composite_I_mean_std,FeatSel_F1_mean,FeatSel_F1_std,FeatSel_F1_mean_std,RuleSparsity_mean,RuleSparsity_std,RuleSparsity_mean_std,MeanDisplayedConditions_mean,MeanDisplayedConditions_std,MeanDisplayedConditions_mean_std,InteractDetect_mean,InteractDetect_std,InteractDetect_mean_std,Direction_mean,Direction_std,Direction_mean_std,RMSE_mean,RMSE_std,RMSE_mean_std,R2_mean,R2_std,R2_mean_std,Accuracy_mean,Accuracy_std,Accuracy_mean_std,F1_mean,F1_std,F1_mean_std,MaskMean_mean,MaskMean_std,MaskMean_mean_std,MaskEntropy_mean,MaskEntropy_std,MaskEntropy_mean_std,DisplayedTopRuleConditions_mean,DisplayedTopRuleConditions_std,DisplayedTopRuleConditions_mean_std,NuMean_mean,NuMean_std,NuMean_mean_std,TemplateScaleMean_mean,TemplateScaleMean_std,TemplateScaleMean_mean_std,ParamsTrainable_mean,ParamsTrainable_std,ParamsTrainable_mean_std,PaperScore
11,rule_count,SMTE-R128,R=128,SMTE,128,0.0001,25,0.7454,0.3602,0.745±0.360,1.9400,1.0540,1.940±1.054,3.0675,0.1512,3.068±0.151,148.6800,6.6000,148.680±6.600,0.0450,0.0284,0.045±0.028,0.8863,0.0387,0.886±0.039,1.0000,0.0000,1.000±0.000,0.8821,0.0382,0.882±0.038,1.7680,0.5735,1.768±0.574,0.7587,0.1422,0.759±0.142,0.7600,0.2930,0.760±0.293,0.4557,0.0722,0.456±0.072,0.9721,0.0222,0.972±0.022,0.9820,0.0081,0.982±0.008,0.9802,0.0083,0.980±0.008,0.0496,0.0007,0.050±0.001,0.1972,0.0021,0.197±0.002,8.8400,2.8676,8.840±2.868,1.7546,0.0272,1.755±0.027,0.6926,0.0048,0.693±0.005,12060.8000,153.5407,12060.800±153.541,0.8017
14,rule_count,SMTE-R64,R=64,SMTE,64,0.0001,25,0.7235,0.3563,0.724±0.356,2.0200,0.8352,2.020±0.835,3.1084,0.0506,3.108±0.051,150.0000,0.0000,150.000±0.000,0.0481,0.0286,0.048±0.029,0.8739,0.0584,0.874±0.058,1.0000,0.0000,1.000±0.000,0.8443,0.0324,0.844±0.032,2.3360,0.4855,2.336±0.486,0.8040,0.1608,0.804±0.161,0.6800,0.3500,0.680±0.350,0.4953,0.0796,0.495±0.080,0.9675,0.0256,0.967±0.026,0.9832,0.0104,0.983±0.010,0.9817,0.0099,0.982±0.010,0.0498,0.0002,0.050±0.000,0.1980,0.0005,0.198±0.000,11.6800,2.4276,11.680±2.428,1.7652,0.0426,1.765±0.043,0.6923,0.0045,0.692±0.004,6032.0000,77.3655,6032.000±77.365,0.7837
13,rule_count,SMTE-R32,R=32,SMTE,32,0.0001,25,0.6214,0.3516,0.621±0.352,2.6200,0.9274,2.620±0.927,3.0871,0.0468,3.087±0.047,150.0000,0.0000,150.000±0.000,0.0693,0.0410,0.069±0.041,0.8833,0.0584,0.883±0.058,1.0000,0.0000,1.000±0.000,0.8160,0.0283,0.816±0.028,2.7600,0.4243,2.760±0.424,0.8267,0.1337,0.827±0.134,0.7600,0.2930,0.760±0.293,0.5808,0.1006,0.581±0.101,0.9561,0.0338,0.956±0.034,0.9795,0.0179,0.979±0.018,0.9789,0.0157,0.979±0.016,0.0502,0.0002,0.050±0.000,0.1991,0.0006,0.199±0.001,13.8000,2.1213,13.800±2.121,1.7920,0.0799,1.792±0.080,0.6929,0.0051,0.693±0.005,3017.6000,39.2779,3017.600±39.278,0.7261
12,rule_count,SMTE-R16,R=16,SMTE,16,0.0001,25,0.2292,0.3890,0.229±0.389,3.4200,0.9755,3.420±0.976,3.1032,0.0651,3.103±0.065,149.9600,0.2000,149.960±0.200,0.1110,0.0670,0.111±0.067,0.8835,0.0518,0.883±0.052,1.0000,0.0000,1.000±0.000,0.7829,0.0395,0.783±0.040,3.2560,0.5931,3.256±0.593,0.8987,0.1227,0.899±0.123,0.7200,0.3253,0.720±0.325,0.7279,0.1271,0.728±0.127,0.9333,0.0537,0.933±0.054,0.9778,0.0136,0.978±0.014,0.9764,0.0120,0.976±0.012,0.0508,0.0003,0.051±0.000,0.2007,0.0009,0.201±0.001,16.2800,2.9654,16.280±2.965,1.8433,0.1339,1.843±0.134,0.6911,0.0054,0.691±0.005,1510.4000,20.2340,1510.400±20.234,0.4909


,Group,Config,Label,Family,Rules,MaskL1,Runs,PerfScore_mean,PerfScore_std,PerfScore_mean_std,Rank_mean,Rank_std,Rank_mean_std,TrainTime_mean,TrainTime_std,TrainTime_mean_std,EpochsRun_mean,EpochsRun_std,EpochsRun_mean_std,BestValLoss_mean,BestValLoss_std,BestValLoss_mean_std,Composite_I_mean,Composite_I_std,Composite_I_mean_std,FeatSel_F1_mean,FeatSel_F1_std,FeatSel_F1_mean_std,RuleSparsity_mean,RuleSparsity_std,RuleSparsity_mean_std,MeanDisplayedConditions_mean,MeanDisplayedConditions_std,MeanDisplayedConditions_mean_std,InteractDetect_mean,InteractDetect_std,InteractDetect_mean_std,Direction_mean,Direction_std,Direction_mean_std,RMSE_mean,RMSE_std,RMSE_mean_std,R2_mean,R2_std,R2_mean_std,Accuracy_mean,Accuracy_std,Accuracy_mean_std,F1_mean,F1_std,F1_mean_std,MaskMean_mean,MaskMean_std,MaskMean_mean_std,MaskEntropy_mean,MaskEntropy_std,MaskEntropy_mean_std,DisplayedTopRuleConditions_mean,DisplayedTopRuleConditions_std,DisplayedTopRuleConditions_mean_std,NuMean_mean,NuMean_std,NuMean_mean_std,TemplateScaleMean_mean,TemplateScaleMean_std,TemplateScaleMean_mean_std,ParamsTrainable_mean,ParamsTrainable_std,ParamsTrainable_mean_std,PaperScore
8,mask_penalty,SMTE-lambda-1em03,$\lambda_m=0.001$,SMTE,32,0.0010,25,0.5267,0.4171,0.527±0.417,2.4400,1.0540,2.440±1.054,3.0936,0.0610,3.094±0.061,150.0000,0.0000,150.000±0.000,0.0660,0.0409,0.066±0.041,0.9185,0.0610,0.919±0.061,1.0000,0.0000,1.000±0.000,0.9755,0.0263,0.975±0.026,0.3680,0.3945,0.368±0.394,0.8333,0.1727,0.833±0.173,0.7200,0.3253,0.720±0.325,0.5774,0.0920,0.577±0.092,0.9564,0.0341,0.956±0.034,0.9815,0.0148,0.981±0.015,0.9802,0.0138,0.980±0.014,0.0495,0.0001,0.049±0.000,0.1970,0.0002,0.197±0.000,1.8400,1.9723,1.840±1.972,1.7791,0.0763,1.779±0.076,0.6923,0.0060,0.692±0.006,3017.6000,39.2779,3017.600±39.278,0.6834
7,mask_penalty,SMTE-lambda-0e00,$\lambda_m=0$,SMTE,32,0.0000,25,0.5532,0.4154,0.553±0.415,2.3200,1.0091,2.320±1.009,2.9806,0.0418,2.981±0.042,150.0000,0.0000,150.000±0.000,0.0677,0.0472,0.068±0.047,0.8473,0.0698,0.847±0.070,1.0000,0.0000,1.000±0.000,0.7333,0.0000,0.733±0.000,4.0000,0.0000,4.000±0.000,0.8400,0.1443,0.840±0.144,0.6400,0.3686,0.640±0.369,0.5656,0.0751,0.566±0.075,0.9593,0.0311,0.959±0.031,0.9835,0.0111,0.983±0.011,0.9821,0.0099,0.982±0.010,0.1204,0.0003,0.120±0.000,0.3671,0.0007,0.367±0.001,20.0000,0.0000,20.000±0.000,1.7835,0.0670,1.783±0.067,0.6922,0.0053,0.692±0.005,3017.6000,39.2779,3017.600±39.278,0.6709
10,mask_penalty,SMTE-lambda-1em05,$\lambda_m=1e-05$,SMTE,32,0.0000,25,0.4879,0.4672,0.488±0.467,2.5200,1.1680,2.520±1.168,3.0832,0.0461,3.083±0.046,150.0000,0.0000,150.000±0.000,0.0752,0.0589,0.075±0.059,0.8576,0.0630,0.858±0.063,1.0000,0.0000,1.000±0.000,0.7339,0.0027,0.734±0.003,3.9920,0.0400,3.992±0.040,0.8307,0.2156,0.831±0.216,0.7200,0.3253,0.720±0.325,0.5763,0.0861,0.576±0.086,0.9578,0.0316,0.958±0.032,0.9797,0.0159,0.980±0.016,0.9788,0.0148,0.979±0.015,0.0576,0.0017,0.058±0.002,0.2184,0.0046,0.218±0.005,19.9600,0.2000,19.960±0.200,1.7915,0.0805,1.791±0.080,0.6934,0.0062,0.693±0.006,3017.6000,39.2779,3017.600±39.278,0.6358
9,mask_penalty,SMTE-lambda-1em04,$\lambda_m=0.0001$,SMTE,32,0.0001,25,0.4573,0.4227,0.457±0.423,2.7200,1.0516,2.720±1.052,3.0860,0.0442,3.086±0.044,150.0000,0.0000,150.000±0.000,0.0687,0.0419,0.069±0.042,0.8771,0.0621,0.877±0.062,1.0000,0.0000,1.000±0.000,0.8059,0.0418,0.806±0.042,2.9120,0.6274,2.912±0.627,0.8080,0.1785,0.808±0.179,0.7600,0.2930,0.760±0.293,0.5688,0.0777,0.569±0.078,0.9595,0.0298,0.959±0.030,0.9790,0.0144,0.979±0.014,0.9781,0.0137,0.978±0.014,0.0502,0.0002,0.050±0.000,0.1992,0.0006,0.199±0.001,14.5600,3.1369,14.560±3.137,1.7909,0.0857,1.791±0.086,0.6925,0.0054,0.693±0.005,3017.6000,39.2779,3017.600±39.278,0.6252


## 11. Baseline-relative deltas for architecture ablations

In [12]:
baseline = arch_summary[arch_summary["Config"] == "SMTE-reference"].iloc[0].to_dict()

delta_rows = []
for _, row in arch_summary.iterrows():
    d = row.to_dict()
    d["DeltaPerfScore"] = row["PerfScore_mean"] - baseline["PerfScore_mean"]
    d["DeltaComposite_I"] = row["Composite_I_mean"] - baseline["Composite_I_mean"]
    d["DeltaFeatSel_F1"] = row["FeatSel_F1_mean"] - baseline["FeatSel_F1_mean"]
    d["DeltaRuleSparsity"] = row["RuleSparsity_mean"] - baseline["RuleSparsity_mean"]
    d["DeltaMaskEntropy"] = row["MaskEntropy_mean"] - baseline["MaskEntropy_mean"]
    d["DeltaTrainTime"] = row["TrainTime_mean"] - baseline["TrainTime_mean"]
    delta_rows.append(d)

arch_delta = pd.DataFrame(delta_rows)
arch_delta.to_csv(TABLE_DIR / "paper_table_architecture_ablation_deltas.csv", index=False)
display(arch_delta[["Config", "Label", "DeltaPerfScore", "DeltaComposite_I", "DeltaFeatSel_F1", "DeltaRuleSparsity", "DeltaMaskEntropy", "DeltaTrainTime"]])

,Config,Label,DeltaPerfScore,DeltaComposite_I,DeltaFeatSel_F1,DeltaRuleSparsity,DeltaMaskEntropy,DeltaTrainTime
0,NoMaskPenalty,No mask penalty,0.0522,-0.0056,0.0000,-0.0853,0.1681,-0.3669
1,GMTE-template,Gaussian template,0.0029,0.0003,-0.0200,0.0240,-0.0002,-0.5658
2,SMTE-reference,Reference SMTE,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,FixedScales,Fixed scales,-0.0915,-0.0061,-0.0100,0.0085,0.0002,-0.3371
4,FixedThresholds,Fixed thresholds,-0.0985,-0.0217,0.0000,-0.0085,0.0007,-0.3133
5,NoSignedEvidence,No signed evidence,-0.3057,-0.0500,-0.0100,-0.0400,0.0021,-0.4356
6,NoLogMargin,No log-margin,-0.4387,-0.0693,-0.0700,-0.0688,0.0721,-0.5211


## 12. Paper-facing compact tables

In [13]:
def compact_paper_table(df: pd.DataFrame, cols: List[Tuple[str, str]]) -> pd.DataFrame:
    out = pd.DataFrame()
    for source, target in cols:
        if source in df.columns:
            out[target] = df[source]
        else:
            out[target] = np.nan
    return out


paper_arch = compact_paper_table(
    arch_summary,
    [
        ("Label", "Ablation"),
        ("Family", "Template"),
        ("PerfScore_mean_std", "Perf."),
        ("Rank_mean_std", "Rank"),
        ("Composite_I_mean_std", "Interp."),
        ("FeatSel_F1_mean_std", "F1"),
        ("RuleSparsity_mean_std", "Sparse"),
        ("MaskEntropy_mean_std", "Entropy"),
        ("TrainTime_mean_std", "Time"),
        ("Runs", "Runs"),
    ],
)
paper_arch.to_csv(TABLE_DIR / "paper_compact_architecture_ablations.csv", index=False)

paper_rule = compact_paper_table(
    rule_summary.sort_values("Rules"),
    [
        ("Rules", "Rules"),
        ("PerfScore_mean_std", "Perf."),
        ("Composite_I_mean_std", "Interp."),
        ("FeatSel_F1_mean_std", "F1"),
        ("RuleSparsity_mean_std", "Sparse"),
        ("MaskEntropy_mean_std", "Entropy"),
        ("TrainTime_mean_std", "Time"),
        ("ParamsTrainable_mean_std", "Params"),
        ("Runs", "Runs"),
    ],
)
paper_rule.to_csv(TABLE_DIR / "paper_compact_rule_count_sensitivity.csv", index=False)

paper_mask = compact_paper_table(
    mask_summary_df.sort_values("MaskL1"),
    [
        ("MaskL1", "Mask penalty"),
        ("PerfScore_mean_std", "Perf."),
        ("Composite_I_mean_std", "Interp."),
        ("RuleSparsity_mean_std", "Sparse"),
        ("MeanDisplayedConditions_mean_std", "Cond."),
        ("MaskEntropy_mean_std", "Entropy"),
        ("TrainTime_mean_std", "Time"),
        ("Runs", "Runs"),
    ],
)
paper_mask.to_csv(TABLE_DIR / "paper_compact_mask_penalty_sensitivity.csv", index=False)

display(paper_arch)
display(paper_rule)
display(paper_mask)

,Ablation,Template,Perf.,Rank,Interp.,F1,Sparse,Entropy,Time,Runs
4,No mask penalty,SMTE,0.877±0.152,2.520±1.212,0.875±0.047,1.000±0.000,0.733±0.000,0.367±0.001,2.978±0.035,25
2,Gaussian template,GMTE,0.827±0.191,3.260±1.473,0.881±0.077,0.980±0.100,0.843±0.028,0.199±0.001,2.779±0.040,25
6,Reference SMTE,SMTE,0.824±0.184,3.280±1.634,0.881±0.068,1.000±0.000,0.819±0.042,0.199±0.001,3.345±1.248,25
0,Fixed scales,SMTE,0.733±0.187,4.500±1.500,0.875±0.067,0.990±0.050,0.827±0.031,0.199±0.001,3.008±0.048,25
1,Fixed thresholds,SMTE,0.726±0.206,4.620±1.285,0.859±0.054,1.000±0.000,0.810±0.038,0.200±0.001,3.031±0.071,25
5,No signed evidence,SMTE,0.519±0.475,4.800±2.466,0.831±0.063,0.990±0.050,0.779±0.037,0.201±0.001,2.909±0.119,25
3,No log-margin,SMTE,0.386±0.459,5.020±2.523,0.812±0.130,0.930±0.170,0.750±0.020,0.271±0.014,2.824±0.185,25


,Rules,Perf.,Interp.,F1,Sparse,Entropy,Time,Params,Runs
12,16,0.229±0.389,0.883±0.052,1.000±0.000,0.783±0.040,0.201±0.001,3.103±0.065,1510.400±20.234,25
13,32,0.621±0.352,0.883±0.058,1.000±0.000,0.816±0.028,0.199±0.001,3.087±0.047,3017.600±39.278,25
14,64,0.724±0.356,0.874±0.058,1.000±0.000,0.844±0.032,0.198±0.000,3.108±0.051,6032.000±77.365,25
11,128,0.745±0.360,0.886±0.039,1.000±0.000,0.882±0.038,0.197±0.002,3.068±0.151,12060.800±153.541,25


,Mask penalty,Perf.,Interp.,Sparse,Cond.,Entropy,Time,Runs
7,0.0000,0.553±0.415,0.847±0.070,0.733±0.000,4.000±0.000,0.367±0.001,2.981±0.042,25
10,0.0000,0.488±0.467,0.858±0.063,0.734±0.003,3.992±0.040,0.218±0.005,3.083±0.046,25
9,0.0001,0.457±0.423,0.877±0.062,0.806±0.042,2.912±0.627,0.199±0.001,3.086±0.044,25
8,0.0010,0.527±0.417,0.919±0.061,0.975±0.026,0.368±0.394,0.197±0.000,3.094±0.061,25


## 13. Figures: architecture ablations

In [14]:
plot_cols = [
    "PerfScore_mean",
    "Composite_I_mean",
    "FeatSel_F1_mean",
    "RuleSparsity_mean",
    "InteractDetect_mean",
    "Direction_mean",
    "MaskEntropy_mean",
]
heat_df = arch_summary.set_index("Label")[plot_cols].copy()
heat_df.columns = ["Perf.", "Interp.", "FeatF1", "Sparse", "Interact", "Direction", "Entropy"]

fig, ax = plt.subplots(figsize=(9.0, 4.6))
sns.heatmap(
    heat_df,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "mean score/value"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
savefig("figure_1_architecture_ablation_heatmap")

In [15]:
delta_plot = arch_delta[arch_delta["Config"] != "SMTE-reference"].copy()
delta_plot = delta_plot.sort_values("DeltaComposite_I")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), sharey=True)

for ax, metric, xlabel, color in [
    (axes[0], "DeltaPerfScore", "Δ normalized performance (relative to reference SMTE)", "#2563EB"),
    (axes[1], "DeltaComposite_I", "Δ interpretability score (relative to reference SMTE)", "#7C3AED"),
]:
    y = np.arange(len(delta_plot))
    vals = delta_plot[metric].values

    ax.axvline(0, color="black", linewidth=1.0)
    ax.hlines(y=y, xmin=0, xmax=vals, color=color, alpha=0.75, linewidth=2)
    ax.scatter(vals, y, s=70, color=color, edgecolor="black", linewidth=0.6, zorder=3)

    ax.set_yticks(y)
    ax.set_yticklabels(delta_plot["Label"])
    ax.set_xlabel(xlabel)

savefig("figure_2_architecture_delta_lollipop")

In [16]:
fig, ax = plt.subplots(figsize=(6.8, 5.0))

for _, row in arch_summary.iterrows():
    label = row["Label"]
    cfg_name = row["Config"]
    color = MODEL_COLORS.get(cfg_name, "#64748B")

    size = 90 + 240 * (
        (row["TrainTime_mean"] - arch_summary["TrainTime_mean"].min())
        / (arch_summary["TrainTime_mean"].max() - arch_summary["TrainTime_mean"].min() + 1e-12)
    )

    ax.scatter(
        row["Composite_I_mean"],
        row["PerfScore_mean"],
        s=size,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        alpha=0.85,
    )
    ax.text(row["Composite_I_mean"] + 0.004, row["PerfScore_mean"] + 0.004, label, fontsize=8)

ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Composite interpretability")
ax.set_ylabel("Dataset-normalized performance")
savefig("figure_3_ablation_frontier")

## 14. Figures: sensitivity curves

In [17]:
rule_plot = rule_summary.sort_values("Rules").copy()

fig, ax1 = plt.subplots(figsize=(7.0, 4.6))

ax1.plot(
    rule_plot["Rules"],
    rule_plot["PerfScore_mean"],
    marker="o",
    linewidth=2.2,
    label="Performance",
    color=SENS_COLORS["PerfScore"],
)
ax1.plot(
    rule_plot["Rules"],
    rule_plot["Composite_I_mean"],
    marker="s",
    linewidth=2.2,
    label="Interpretability",
    color=SENS_COLORS["Composite_I"],
)
ax1.plot(
    rule_plot["Rules"],
    rule_plot["RuleSparsity_mean"],
    marker="^",
    linewidth=2.2,
    label="Rule sparsity",
    color=SENS_COLORS["RuleSparsity"],
)
ax1.set_xscale("log", base=2)
ax1.set_xticks(rule_plot["Rules"])
ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax1.set_ylim(0, 1.05)
ax1.set_xlabel("Number of rules R")
ax1.set_ylabel("score")

ax2 = ax1.twinx()
ax2.plot(
    rule_plot["Rules"],
    rule_plot["TrainTime_mean"],
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label="Train time",
    color=SENS_COLORS["TrainTime"],
)
ax2.set_ylabel("train time, seconds")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=True, loc="best")

savefig("figure_4_rule_count_sensitivity")

In [18]:
mask_plot = mask_summary_df.sort_values("MaskL1").copy()
mask_plot["MaskL1_plot"] = mask_plot["MaskL1"].replace(0.0, 1e-7)

fig, ax1 = plt.subplots(figsize=(7.2, 4.8))

ax1.plot(
    mask_plot["MaskL1_plot"],
    mask_plot["PerfScore_mean"],
    marker="o",
    linewidth=2.2,
    label="Performance",
    color=SENS_COLORS["PerfScore"],
)
ax1.plot(
    mask_plot["MaskL1_plot"],
    mask_plot["Composite_I_mean"],
    marker="s",
    linewidth=2.2,
    label="Interpretability",
    color=SENS_COLORS["Composite_I"],
)
ax1.plot(
    mask_plot["MaskL1_plot"],
    mask_plot["RuleSparsity_mean"],
    marker="^",
    linewidth=2.2,
    label="Rule sparsity",
    color=SENS_COLORS["RuleSparsity"],
)

ax1.set_xscale("log")
ax1.set_xticks(mask_plot["MaskL1_plot"])
ax1.set_xticklabels(["0" if x == 1e-7 else f"{x:g}" for x in mask_plot["MaskL1_plot"]])
ax1.set_ylim(0, 1.05)
ax1.set_xlabel("Mask penalty λm")
ax1.set_ylabel("score")

ax2 = ax1.twinx()
ax2.plot(
    mask_plot["MaskL1_plot"],
    mask_plot["MaskEntropy_mean"],
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label="Mask entropy",
    color=SENS_COLORS["MaskEntropy"],
)
ax2.set_ylabel("mask entropy")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=True, loc="best")

savefig("figure_5_mask_penalty_sensitivity")

In [19]:
sens_rows = []

for _, row in rule_summary.iterrows():
    sens_rows.append(
        {
            "Family": "Rule count",
            "Setting": f"R={int(row['Rules'])}",
            "Performance": row["PerfScore_mean"],
            "Interpretability": row["Composite_I_mean"],
            "Sparsity": row["RuleSparsity_mean"],
            "Entropy": row["MaskEntropy_mean"],
        }
    )

for _, row in mask_summary_df.sort_values("MaskL1").iterrows():
    lam = row["MaskL1"]
    sens_rows.append(
        {
            "Family": "Mask penalty",
            "Setting": f"λ={lam:g}",
            "Performance": row["PerfScore_mean"],
            "Interpretability": row["Composite_I_mean"],
            "Sparsity": row["RuleSparsity_mean"],
            "Entropy": row["MaskEntropy_mean"],
        }
    )

sens_tile = pd.DataFrame(sens_rows)
sens_long = sens_tile.melt(id_vars=["Family", "Setting"], var_name="Metric", value_name="Value")
sens_long["Row"] = sens_long["Family"] + ": " + sens_long["Setting"]

tile = sens_long.pivot(index="Row", columns="Metric", values="Value")

fig, ax = plt.subplots(figsize=(7.6, 5.2))
sns.heatmap(
    tile,
    cmap="mako",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "mean value"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
savefig("figure_6_sensitivity_tile_map")

## 15. Training-curve figure for representative ablations

In [20]:
if not df_history.empty:
    representative_seed = SEEDS[0]
    representative_dataset = "Hierarchical" if "Hierarchical" in DATASET_NAMES else DATASET_NAMES[0]
    representative_configs = [
        "SMTE-reference",
        "NoSignedEvidence",
        "NoLogMargin",
        "NoMaskPenalty",
        "GMTE-template",
    ]

    hplot = df_history[
        (df_history["Seed"] == representative_seed)
        & (df_history["Dataset"] == representative_dataset)
        & (df_history["Config"].isin(representative_configs))
    ].copy()

    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))

    for cfg_name, g in hplot.groupby("Config"):
        color = MODEL_COLORS.get(cfg_name, None)
        axes[0].plot(g["Epoch"], g["ValLoss"], label=cfg_name, linewidth=2, color=color)
        axes[1].plot(g["Epoch"], g["MaskMean"], label=cfg_name, linewidth=2, color=color)

    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("validation loss")
    axes[0].legend(frameon=True)

    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mean mask value")
    axes[1].legend(frameon=True)

    savefig("figure_7_representative_training_diagnostics")

## 16. Simple Typst snippets for manuscript insertion

In [21]:
def typst_escape(x: Any) -> str:
    s = str(x)
    s = s.replace("\\", "\\\\")
    s = s.replace("[", "\\[")
    s = s.replace("]", "\\]")
    return s


def typst_table_from_df(
    df: pd.DataFrame,
    caption: str,
    label: str,
    max_rows: Optional[int] = None,
) -> str:
    if max_rows is not None:
        df = df.head(max_rows).copy()

    ncols = len(df.columns)
    colspec = ", ".join(["1fr"] * ncols)

    header = ",\n      ".join([f"[*{typst_escape(c)}*]" for c in df.columns])

    rows = []
    for _, row in df.iterrows():
        cells = []
        for c in df.columns:
            val = row[c]
            if isinstance(val, float):
                if np.isnan(val):
                    sval = "nan"
                else:
                    sval = f"{val:.3f}"
            else:
                sval = typst_escape(val)
            cells.append(f"[{sval}]")
        rows.append(",\n      ".join(cells))

    body = ",\n      ".join([header] + rows)

    return f"""#here-table(
  compact-table[
    #table(
      columns: ({colspec}),
      inset: 2pt,
      align: center + horizon,
      table.header({header}),
      {",\n      ".join(rows)}
    )
  ],
  [{typst_escape(caption)}]
) <{label}>
"""


typst_arch_df = paper_arch.copy()
typst_rule_df = paper_rule.copy()
typst_mask_df = paper_mask.copy()

typst_snippets = []

typst_snippets.append("// Architecture ablation table\n")
typst_snippets.append(
    typst_table_from_df(
        typst_arch_df,
        "Controlled architecture ablations over synthetic datasets. Performance is dataset-normalized; interpretability is the composite score.",
        "tab-ablation-architecture",
    )
)

typst_snippets.append("\n// Rule-count sensitivity table\n")
typst_snippets.append(
    typst_table_from_df(
        typst_rule_df,
        "Rule-count sensitivity for SMTE. Increasing R changes capacity, readability, and runtime.",
        "tab-ablation-rule-count",
    )
)

typst_snippets.append("\n// Mask-penalty sensitivity table\n")
typst_snippets.append(
    typst_table_from_df(
        typst_mask_df,
        "Mask-penalty sensitivity for SMTE. Larger penalties encourage sparser displayed rules.",
        "tab-ablation-mask-penalty",
    )
)

typst_snippets.append(
    """
// Figure block for ablation protocol
#here-figure(
  grid(
    columns: (1fr, 1fr),
    gutter: 6pt,
    paper-image("outputs_ablation_paper/figures/figure_1_architecture_ablation_heatmap.png", width: 100%),
    paper-image("outputs_ablation_paper/figures/figure_2_architecture_delta_lollipop.png", width: 100%),
  ),
  [Controlled architecture ablations. Left: mean diagnostic scores. Right: change relative to reference SMTE.]
) <fig-ablation-architecture>

#here-figure(
  grid(
    columns: (1fr, 1fr),
    gutter: 6pt,
    paper-image("outputs_ablation_paper/figures/figure_4_rule_count_sensitivity.png", width: 100%),
    paper-image("outputs_ablation_paper/figures/figure_5_mask_penalty_sensitivity.png", width: 100%),
  ),
  [Sensitivity curves for rule count and mask penalty.]
) <fig-ablation-sensitivity>
"""
)

typst_text = "\n".join(typst_snippets)

with open(OUTPUT_DIR / "typst_ablation_snippets.typ", "w", encoding="utf-8") as f:
    f.write(typst_text)

print(typst_text[:4000])
print(f"\nTypst snippets saved to: {OUTPUT_DIR / 'typst_ablation_snippets.typ'}")

// Architecture ablation table

#here-table(
  compact-table[
    #table(
      columns: (1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr),
      inset: 2pt,
      align: center + horizon,
      table.header([*Ablation*],
      [*Template*],
      [*Perf.*],
      [*Rank*],
      [*Interp.*],
      [*F1*],
      [*Sparse*],
      [*Entropy*],
      [*Time*],
      [*Runs*]),
      [No mask penalty],
      [SMTE],
      [0.877±0.152],
      [2.520±1.212],
      [0.875±0.047],
      [1.000±0.000],
      [0.733±0.000],
      [0.367±0.001],
      [2.978±0.035],
      [25],
      [Gaussian template],
      [GMTE],
      [0.827±0.191],
      [3.260±1.473],
      [0.881±0.077],
      [0.980±0.100],
      [0.843±0.028],
      [0.199±0.001],
      [2.779±0.040],
      [25],
      [Reference SMTE],
      [SMTE],
      [0.824±0.184],
      [3.280±1.634],
      [0.881±0.068],
      [1.000±0.000],
      [0.819±0.042],
      [0.199±0.001],
      [3.345±1.248],
      [25],
      [Fixed scales],
    

## 17. Manifest

In [22]:
env_table = pd.DataFrame(
    [
        {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "processor": platform.processor(),
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "device": str(DEVICE),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "sklearn": __import__("sklearn").__version__,
            "seeds": str(SEEDS),
            "datasets": str(DATASET_NAMES),
            "quick_mode": QUICK_MODE,
            "n_main": N_MAIN,
            "d_main": D_MAIN,
            "default_n_rules": DEFAULT_N_RULES,
            "epochs": EPOCHS,
            "patience": PATIENCE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "default_mask_l1": DEFAULT_MASK_L1,
            "rule_beta": RULE_BETA,
            "gradient_n_samples": GRADIENT_N_SAMPLES,
            "rule_counts": str(RULE_COUNTS),
            "mask_l1_values": str(MASK_L1_VALUES),
        }
    ]
)
env_table.to_csv(TABLE_DIR / "table_ablation_environment.csv", index=False)

manifest = {
    "output_dir": str(OUTPUT_DIR),
    "quick_mode": QUICK_MODE,
    "seeds": SEEDS,
    "datasets": DATASET_NAMES,
    "device": str(DEVICE),
    "n_main": N_MAIN,
    "d_main": D_MAIN,
    "default_n_rules": DEFAULT_N_RULES,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "default_mask_l1": DEFAULT_MASK_L1,
    "rule_beta": RULE_BETA,
    "rule_counts": RULE_COUNTS,
    "mask_l1_values": MASK_L1_VALUES,
    "n_successful_runs": int(len(df_raw)),
    "n_failed_runs": int(len(df_failed)),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*.csv")]),
    "figures": sorted([p.name for p in FIG_DIR.glob("*.png")]),
    "extra_files": [
        "typst_ablation_snippets.typ",
        "suggested_manuscript_ablation_text.txt",
    ],
}

with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nFiles saved:")
for fname in manifest["tables"]:
    print(f"  tables/{fname}")

for fname in manifest["figures"]:
    print(f"  figures/{fname}")

print("\nManifest:")
print(json.dumps(manifest, indent=2))


Files saved:
  tables/paper_compact_architecture_ablations.csv
  tables/paper_compact_mask_penalty_sensitivity.csv
  tables/paper_compact_rule_count_sensitivity.csv
  tables/paper_table_architecture_ablation_deltas.csv
  tables/paper_table_architecture_ablations.csv
  tables/paper_table_mask_penalty_sensitivity.csv
  tables/paper_table_rule_count_sensitivity.csv
  tables/table_ablation_configurations.csv
  tables/table_ablation_dataset_specification.csv
  tables/table_ablation_environment.csv
  tables/table_ablation_extracted_rules.csv
  tables/table_ablation_failed_runs.csv
  tables/table_ablation_ranked_results.csv
  tables/table_ablation_raw_results.csv
  tables/table_ablation_summary_by_config.csv
  tables/table_ablation_summary_by_dataset.csv
  tables/table_ablation_training_curves.csv
  figures/figure_1_architecture_ablation_heatmap.png
  figures/figure_2_architecture_delta_lollipop.png
  figures/figure_3_ablation_frontier.png
  figures/figure_4_rule_count_sensitivity.png
  figu